### Machine Learning Pipelines in Scikit-learn

A machine-learning workflow usually contains several steps before and after model training:

1. Handle missing values.
2. Encode categorical variables.
3. Scale numerical features when needed.
4. Select useful features.
5. Train a model.
6. Predict and evaluate.

A **Pipeline** chains these steps into one reusable object. Instead of manually applying each transformation, combining arrays, and remembering the correct order, we can call `fit()`, `predict()`, or use cross-validation on the complete workflow.

This notebook builds a Titanic survival-classification workflow using `ColumnTransformer`, preprocessing pipelines, feature selection, and a `DecisionTreeClassifier`.

#### Why use a Pipeline?

- **Prevents data leakage:** during `fit`, each preprocessing step learns only from the training fold.
- **Consistent transformations:** training, testing, cross-validation, and future predictions use the same fitted preprocessing steps.
- **Less manual code:** no need to separately transform every column group and concatenate arrays.
- **Easier cross-validation and tuning:** the entire workflow can be evaluated or tuned as one estimator.
- **Reusable and deployable:** the fitted preprocessing and model can be saved together.

#### Pipeline flow diagram

```text
                 Raw Titanic dataset
                          |
                    Train / Test split
                          |
                 +-------------------+
                 |  ColumnTransformer|
                 +-------------------+
                    /             \
                   /               \
          Numerical columns    Categorical columns
                 |                    |
          SimpleImputer          SimpleImputer
           (median)             (most frequent)
                 |                    |
           MinMaxScaler           OneHotEncoder
                   \                /
                    \              /
                 Combined feature matrix
                          |
                    SelectKBest
                    (chi-square)
                          |
                DecisionTreeClassifier
                          |
                 Prediction / Evaluation
```

The diagram shows the conceptual order. The exact column groups and transformations are defined in the code below.

In [1]:
import numpy as np 
import pandas as pd 

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier

In [3]:
df = pd.read_csv("Datasets/titanic.csv")

In [4]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)

In [6]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [7]:
# train test split 
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['Survived']), df['Survived'], test_size=0.2, random_state=42)

In [8]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [9]:
y_train.head()

331    0
733    0
382    0
704    0
813    0
Name: Survived, dtype: int64

### Part 1: Define preprocessing steps

The original notebook demonstrates a sequence of separate transformers. A key detail is that `ColumnTransformer` places transformed columns first and then appends any `remainder` columns. Therefore, selecting columns by numeric position in a later transformer can accidentally select different columns than intended.

To make the workflow reliable, the updated implementation below uses **column names** and combines related operations in nested `Pipeline` objects.

#### Numerical preprocessing

The numerical pipeline:

1. Fills missing values with the median.
2. Scales the resulting values with `MinMaxScaler`.

Median imputation is less sensitive to extreme values than mean imputation. Scaling is included to demonstrate a common preprocessing step; decision trees generally do not require feature scaling.

In [10]:
# Numerical preprocessing: impute missing values, then scale
numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

#### Categorical preprocessing

The categorical pipeline:

1. Fills missing categories with the most frequent value.
2. Applies one-hot encoding.
3. Uses `handle_unknown='ignore'` so categories not seen during fitting do not cause a transform-time error.

The encoded categories are nominal, so one-hot encoding avoids introducing an artificial order.

In [11]:
# Categorical preprocessing: impute missing values, then one-hot encode
categorical_features = ['Sex', 'Embarked']

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

#### Build the `ColumnTransformer`

`ColumnTransformer` applies the numerical pipeline to numerical columns and the categorical pipeline to categorical columns. The outputs are combined into a single feature matrix.

`remainder='drop'` means any column not listed in either group is excluded. If you want to retain unlisted columns unchanged, use `remainder='passthrough'`.

In [12]:
# Apply the correct preprocessing pipeline to each named column group
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ],
    remainder='drop'
)

### Part 2: Add feature selection and the model

#### Feature selection with `SelectKBest`

`SelectKBest(score_func=chi2, k=8)` keeps the eight features with the highest chi-square scores.

The chi-square feature score is designed for non-negative feature values and a categorical target. The MinMaxScaler in the numerical preprocessing branch produces non-negative values; one-hot encoded values are also non-negative.

Feature selection is fitted as part of the pipeline, so it is learned only from the training data (or the training fold during cross-validation).

In [13]:
# Select the 8 best features using the chi-square score
feature_selection = SelectKBest(score_func=chi2, k=8)

In [15]:
# Classification model
model = DecisionTreeClassifier(random_state=42)

### 14. Choose the classification algorithm

`DecisionTreeClassifier` is the estimator used to predict whether a passenger survived.

A decision tree does not require feature scaling, but scaling is kept in this notebook to demonstrate how multiple preprocessing operations can be chained. For a tree model, you could omit the scaler without usually harming performance.

In [14]:
# Classification model
model = DecisionTreeClassifier(random_state=42)

#### Create Pipeline

### Part 3: Create and train the Pipeline

#### Create the complete pipeline

The complete workflow contains three stages:

1. `preprocessor`: imputes, scales, and encodes the selected columns.
2. `feature_selection`: selects the top eight features.
3. `classifier`: trains the decision tree.

When `pipe.fit(X_train, y_train)` is called, each stage is fitted in order. During `pipe.predict(X_test)`, the already-fitted preprocessing and feature-selection steps transform the test data before the classifier predicts.

In [16]:
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('feature_selection', feature_selection),
    ('classifier', model)
])

#### Alternative: `make_pipeline()`

`make_pipeline()` creates a pipeline when you do not need to provide custom step names. It automatically generates names from the estimator classes.

Use `Pipeline([...])` when you want explicit, meaningful step names or need to refer to steps by those names.


pipe = make_pipeline(preprocessor, feature_selection, model)


In [18]:
# train 
pipe.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Pclass', 'Age', 'SibSp',
                                                   'Parch', 'Fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sex', 'Embarked'])])),
                ('feature_selection',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x000002251B3F1C60>)),
                ('classifier', DecisionTreeClassifier(random_state=42))])

In [19]:
# Attributes 
pipe.named_steps

{'preprocessor': ColumnTransformer(transformers=[('num',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='median')),
                                                  ('scaler', MinMaxScaler())]),
                                  ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']),
                                 ('cat',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='most_frequent')),
                                                  ('onehot',
                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                  ['Sex', 'Embarked'])]),
 'feature_selection': SelectKBest(k=8, score_func=<function chi2 at 0x000002251B3F1C60>),
 'classifier': DecisionTreeClassifier(random_state=42)}

In [20]:
# Median learned for numerical columns (Pclass, Age, SibSp, Parch, Fare)
pipe.named_steps['preprocessor'] \
    .named_transformers_['num'] \
    .named_steps['imputer'].statistics_

array([ 3.    , 28.    ,  0.    ,  0.    , 14.4542])

In [21]:
# Most-frequent values learned for categorical columns (Sex, Embarked)
pipe.named_steps['preprocessor'] \
    .named_transformers_['cat'] \
    .named_steps['imputer'].statistics_

array(['male', 'S'], dtype=object)

In [22]:
# predict 
y_pred = pipe.predict(X_test)

In [23]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

0.7877094972067039

In [24]:
# cross val score 
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

np.float64(0.786673889490791)

In [25]:
# export 
import pickle

pickle.dump(pipe, open('pipe.pkl', 'wb'))

### Why Pipeline is important — summary

| Without a Pipeline | With a Pipeline |
|---|---|
| Manually call each transformer | Call `fit()` once on the workflow |
| Manually keep track of transformed columns | Column transformations are managed in one object |
| Easy to accidentally fit on test data | Cross-validation fits preprocessing within each training fold |
| Manually transform new data in the correct order | Call `predict()` on raw feature columns |
| Save preprocessing and model separately | Save the complete fitted workflow together |

#### When should you use a Pipeline?

Use a pipeline whenever your ML workflow has learned preprocessing steps, such as imputation, scaling, encoding, feature selection, or dimensionality reduction, before the final estimator.

A pipeline does not automatically make every workflow leakage-free: preprocessing must be inside the pipeline, and data-dependent decisions must not be made using the held-out test set.

### Key takeaways

1. A pipeline chains preprocessing, feature selection, and model training.
2. `ColumnTransformer` handles different column groups; `Pipeline` chains steps in sequence.
3. Fit the complete pipeline on training data only.
4. Use the same fitted pipeline for test data, cross-validation, and future predictions.
5. Saving the fitted pipeline preserves the preprocessing logic and the trained estimator together.